# Reproduce the motor-imagery results (Papers 1, 3, 4, 5, 6)
Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. PhysioNet EEGMMIDB is downloaded automatically (~3.4 GB).

In [ ]:
!git clone -q https://github.com/aftab802/eeg-mi-leakage-benchmark
%cd eeg-mi-leakage-benchmark
!pip -q install mne pyedflib edfio
import tensorflow as tf; print('TF', tf.__version__, tf.config.list_physical_devices('GPU'))

In [ ]:
# Paper 3 / Paper 1 Table 7: 1D CNN under the as-executed, L2 and L0 protocols
!python scripts/run_benchmark.py --models conv1d --ablation 2>&1 | grep -v -E 'Downloading|^Epoch|step - |Subject +[0-9]+:' | tail -12

In [ ]:
# Paper 5 / Paper 1: CSP+CNN, subject-wise (L0) and class-stratified record-level (L2)
!python scripts/run_csp_cnn.py --protocol L0 2>&1 | grep -v -E 'Downloading|^Epoch|step - |Subject +[0-9]+:' | tail -12
!python scripts/run_csp_cnn.py --protocol L2 2>&1 | grep -v -E 'Downloading|^Epoch|step - |Subject +[0-9]+:' | tail -12

In [ ]:
# Papers 6 / 1: EEGNet, L0 (verification of 75.16 %)
!python scripts/run_group_aware_eegnet.py 2>&1 | grep -v -E 'Downloading|^Epoch|step - |Subject +[0-9]+:' | tail -12

In [ ]:
# Papers 4 / 1: four-class CNN, original trial-level split (50 and 100 epochs) and subject-wise split
!python scripts/run_fourclass_cnn.py --epochs 50 2>&1 | grep -E 'Epoch +50 |Saved' | tail -3
!python scripts/run_fourclass_cnn.py --epochs 100 2>&1 | grep -E 'Epoch +100 |Saved' | tail -3
!python scripts/run_fourclass_cnn.py --epochs 100 --protocol L0 2>&1 | grep -E 'Epoch +100 |Saved' | tail -3

In [ ]:
import glob, json
rows = []
for f in sorted(glob.glob('results/*.json')):
    r = json.load(open(f))
    items = list(r.items()) if all(isinstance(v, dict) and 'test_accuracy' in v for v in r.values()) else [(f.split('/')[-1], r)]
    for k, v in items:
        row = {k2: v.get(k2) for k2 in ('model', 'protocol', 'test_accuracy', 'train_accuracy', 'final_train_accuracy_first200',
               'n_train', 'n_test', 'train_test_subject_overlap', 'params_total', 'params_trainable_head', 'epochs_run', 'subjects',
               'validation_is_test')}
        row['run'] = k
        rep = v.get('report', {})
        row['recall'] = {c: round(rep[c]['recall'], 4) for c in rep if c not in ('accuracy', 'macro avg', 'weighted avg')}
        row['confusion_matrix'] = v.get('confusion_matrix')
        rows.append(row)
print('RESULTS_JSON_BEGIN'); print(json.dumps(rows)); print('RESULTS_JSON_END')